In [1]:
# ============================================================================
# STEP 1: SETUP TRAINING ENVIRONMENT
# ============================================================================

import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import DataLoader, Subset, Dataset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import time
import json
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("MODEL TRAINING")
print("=" * 70)
print("Student: Sanjay Chaudhary (B01036144)")
print("=" * 70)

# Check hardware
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("\nUsing Apple MPS GPU - this will be faster!")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("\nUsing CUDA GPU!")
else:
    device = torch.device('cpu')
    print("\nNo GPU detected - will run on CPU (slower but it works)")

print(f"Device: {device}")

# Training parameters
BATCH_SIZE = 32
IMG_SIZE = 224
NUM_CLASSES = 5
LR_STAGE1 = 0.001
LR_STAGE2 = 0.0001
EPOCHS_STAGE1 = 5
EPOCHS_STAGE2 = 20
PATIENCE = 7

print("\nTraining Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  Stage 1: {EPOCHS_STAGE1} epochs, LR={LR_STAGE1}")
print(f"  Stage 2: {EPOCHS_STAGE2} epochs, LR={LR_STAGE2}")
print(f"  Early stopping patience: {PATIENCE}")

MODEL TRAINING
Student: Sanjay Chaudhary (B01036144)

Using Apple MPS GPU - this will be faster!
Device: mps

Training Configuration:
  Batch size: 32
  Image size: 224x224
  Stage 1: 5 epochs, LR=0.001
  Stage 2: 20 epochs, LR=0.0001
  Early stopping patience: 7


In [2]:
# ============================================================================
# STEP 2: DEFINE FIXED DATASET CLASS
# ============================================================================

class FixedAPTOSDataset(Dataset):
    """Fix class labels to ICDRSS standard"""
    
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        
        # Map folder names to correct ICDRSS labels
        self.folder_to_label = {
            'Mild': 1,
            'Moderate': 2,
            'No_DR': 0,
            'Proliferate_DR': 4,
            'Severe': 3
        }
        
        # Load raw dataset without transform
        self.raw_dataset = datasets.ImageFolder(root=root_dir, transform=None)
        
        # Store corrected samples
        self.samples = []
        self.labels = []
        
        for img, wrong_label in self.raw_dataset:
            folder_name = self.raw_dataset.classes[wrong_label]
            correct_label = self.folder_to_label[folder_name]
            self.samples.append(img)
            self.labels.append(correct_label)
        
        print(f"Fixed dataset loaded: {len(self.samples)} images")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img = self.samples[idx]
        label = self.labels[idx]
        
        if self.transform:
            img = self.transform(img)
        
        return img, label

In [3]:
# ============================================================================
# STEP 3: DEFINE TRANSFORMATIONS
# ============================================================================

print("\n" + "=" * 70)
print("STEP 3: DEFINING TRANSFORMATIONS")
print("=" * 70)

# Training transforms - with augmentation
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Validation transforms - no augmentation
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("\nTraining transforms (with augmentation):")
print("  - Random horizontal flip")
print("  - Random rotation (+/- 10 degrees)")
print("  - Random brightness and contrast (+/- 20%)")
print("  - Normalize with ImageNet statistics")

print("\nValidation transforms (no augmentation):")
print("  - Resize and normalize only")


STEP 3: DEFINING TRANSFORMATIONS

Training transforms (with augmentation):
  - Random horizontal flip
  - Random rotation (+/- 10 degrees)
  - Random brightness and contrast (+/- 20%)
  - Normalize with ImageNet statistics

Validation transforms (no augmentation):
  - Resize and normalize only


In [4]:
# ============================================================================
# STEP 4: LOAD PREPROCESSED DATA
# ============================================================================

print("\n" + "=" * 70)
print("STEP 4: LOADING PREPROCESSED DATA")
print("=" * 70)

DATA_DIR = "../datasets/aptos_2019/images"

# Load dataset with correct labels
print("\nLoading dataset with correct ICDRSS labels...")
dataset = FixedAPTOSDataset(root_dir=DATA_DIR, transform=train_transform)

# Load split indices from Notebook 02
split_data = torch.load('../models/split_indices.pt', weights_only=False)
train_idx = split_data['train_idx']
val_idx = split_data['val_idx']
class_names = split_data['class_names']

print(f"\nSplit indices loaded:")
print(f"  Training indices: {len(train_idx)}")
print(f"  Validation indices: {len(val_idx)}")
print(f"  Classes: {class_names}")

# Create training dataset (with augmentation)
train_dataset = Subset(dataset, train_idx)

# Create validation dataset (without augmentation)
val_dataset_full = FixedAPTOSDataset(root_dir=DATA_DIR, transform=val_transform)
val_dataset = Subset(val_dataset_full, val_idx)

# Load class weights
class_weights = torch.load('../models/class_weights.pt').to(device)

print(f"\nDatasets created:")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")


STEP 4: LOADING PREPROCESSED DATA

Loading dataset with correct ICDRSS labels...
Fixed dataset loaded: 3662 images

Split indices loaded:
  Training indices: 2929
  Validation indices: 733
  Classes: ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
Fixed dataset loaded: 3662 images

Datasets created:
  Training samples: 2929
  Validation samples: 733


In [5]:
# ============================================================================
# STEP 5: CREATE DATA LOADERS
# ============================================================================

print("\n" + "=" * 70)
print("STEP 5: CREATING DATA LOADERS")
print("=" * 70)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=0,
    pin_memory=True
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

# Test the loaders
images, labels = next(iter(train_loader))
print(f"\nTest batch:")
print(f"  Images shape: {images.shape}")
print(f"  Labels shape: {labels.shape}")
print(f"  Label range: {labels.min()} to {labels.max()}")
print(f"  Pixel range: [{images.min():.3f}, {images.max():.3f}]")
print("\nData loaders working correctly!")


STEP 5: CREATING DATA LOADERS
Training batches: 92
Validation batches: 23

Test batch:
  Images shape: torch.Size([32, 3, 224, 224])
  Labels shape: torch.Size([32])
  Label range: 0 to 4
  Pixel range: [-2.118, 2.640]

Data loaders working correctly!


In [6]:
# ============================================================================
# STEP 6: BUILD THE MODEL
# ============================================================================

print("\n" + "=" * 70)
print("STEP 6: BUILDING THE MODEL")
print("=" * 70)

class DRResNet50(nn.Module):
    """ResNet50 adapted for diabetic retinopathy detection"""
    
    def __init__(self, num_classes=5, pretrained=True):
        super(DRResNet50, self).__init__()
        
        if pretrained:
            print("  Loading pre-trained ResNet50 weights...")
            self.model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        else:
            print("  Initializing ResNet50 from scratch...")
            self.model = models.resnet50(weights=None)
        
        # Replace final layer for 5 classes
        num_features = self.model.fc.in_features
        self.model.fc = nn.Linear(num_features, num_classes)
        self.num_classes = num_classes
    
    def forward(self, x):
        return self.model(x)
    
    def freeze_features(self):
        """Freeze all layers except classifier"""
        for param in self.model.parameters():
            param.requires_grad = False
        for param in self.model.fc.parameters():
            param.requires_grad = True
        print("  Frozen feature extractor, only classifier trainable")
    
    def unfreeze_all(self):
        """Unfreeze everything for fine-tuning"""
        for param in self.model.parameters():
            param.requires_grad = True
        print("  All layers unfrozen for fine-tuning")

# Create model
model = DRResNet50(num_classes=NUM_CLASSES, pretrained=True)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Statistics:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Frozen parameters: {total_params - trainable_params:,}")


STEP 6: BUILDING THE MODEL
  Loading pre-trained ResNet50 weights...

Model Statistics:
  Total parameters: 23,518,277
  Trainable parameters: 23,518,277
  Frozen parameters: 0


In [7]:
# ============================================================================
# STEP 7: DEFINE TRAINING FUNCTIONS
# ============================================================================

print("\n" + "=" * 70)
print("STEP 7: TRAINING FUNCTIONS")
print("=" * 70)

def train_one_epoch(model, loader, criterion, optimizer):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        if (batch_idx + 1) % 20 == 0:
            print(f"    Batch {batch_idx + 1}/{len(loader)} | Loss: {loss.item():.4f}")
    
    return running_loss / len(loader), 100. * correct / total

def validate(model, loader, criterion):
    """Validate the model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return running_loss / len(loader), 100. * correct / total

print("Training functions ready")


STEP 7: TRAINING FUNCTIONS
Training functions ready


In [11]:
# ============================================================================
# STEP 8: STAGE 1 - TRAIN CLASSIFIER ONLY
# ============================================================================

print("\n" + "=" * 70)
print("STAGE 1: Training the Classifier Only")
print("=" * 70)
print("I freeze all pre-trained layers and only train the new classifier.")
print("-" * 50)

# Freeze feature extractor
model.freeze_features()

# Loss function with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer for classifier layer
optimizer = optim.Adam(model.model.fc.parameters(), lr=LR_STAGE1)

# Learning rate scheduler - remove verbose parameter
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.1, patience=3
)

# Track progress
history_stage1 = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(1, EPOCHS_STAGE1 + 1):
    start_time = time.time()
    
    print(f"\nEpoch {epoch}/{EPOCHS_STAGE1}")
    print("-" * 40)
    
    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    
    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion)
    
    # Update scheduler
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save history
    history_stage1['train_loss'].append(train_loss)
    history_stage1['train_acc'].append(train_acc)
    history_stage1['val_loss'].append(val_loss)
    history_stage1['val_acc'].append(val_acc)
    
    epoch_time = time.time() - start_time
    
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print(f"  Time: {epoch_time:.1f}s | LR: {current_lr:.6f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '../models/best_model_stage1.pth')
        print(f"  New best model saved! (Val Acc: {val_acc:.2f}%)")

print(f"\nStage 1 complete! Best validation accuracy: {best_val_acc:.2f}%")


STAGE 1: Training the Classifier Only
I freeze all pre-trained layers and only train the new classifier.
--------------------------------------------------
  Frozen feature extractor, only classifier trainable

Epoch 1/5
----------------------------------------
    Batch 20/92 | Loss: 1.2768
    Batch 40/92 | Loss: 1.1801
    Batch 60/92 | Loss: 1.0525
    Batch 80/92 | Loss: 1.3464
  Train Loss: 1.2408 | Train Acc: 59.78%
  Val Loss:   1.2489 | Val Acc:   62.21%
  Time: 55.1s | LR: 0.001000
  New best model saved! (Val Acc: 62.21%)

Epoch 2/5
----------------------------------------
    Batch 20/92 | Loss: 1.1759
    Batch 40/92 | Loss: 1.2877
    Batch 60/92 | Loss: 1.0639
    Batch 80/92 | Loss: 1.3037
  Train Loss: 1.1295 | Train Acc: 64.02%
  Val Loss:   1.2020 | Val Acc:   64.26%
  Time: 56.0s | LR: 0.001000
  New best model saved! (Val Acc: 64.26%)

Epoch 3/5
----------------------------------------
    Batch 20/92 | Loss: 1.2741
    Batch 40/92 | Loss: 0.9753
    Batch 60/92 |

In [13]:
# ============================================================================
# STEP 9: STAGE 2 - FINE-TUNE ALL LAYERS
# ============================================================================

print("\n" + "=" * 70)
print("STAGE 2: Fine-tuning All Layers")
print("=" * 70)
print("Now I unfreeze everything and train with a lower learning rate.")
print("-" * 50)

# Load best model from stage 1
model.load_state_dict(torch.load('../models/best_model_stage1.pth'))

# Unfreeze all layers
model.unfreeze_all()

# Lower learning rate for fine-tuning
optimizer = optim.Adam(model.parameters(), lr=LR_STAGE2)

# Learning rate scheduler - NO verbose parameter
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.1, patience=3
)

# Track progress
history_stage2 = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = best_val_acc
best_epoch = 0
patience_counter = 0

for epoch in range(1, EPOCHS_STAGE2 + 1):
    start_time = time.time()
    
    print(f"\nEpoch {epoch}/{EPOCHS_STAGE2}")
    print("-" * 40)
    
    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    
    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion)
    
    # Update scheduler
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save history
    history_stage2['train_loss'].append(train_loss)
    history_stage2['train_acc'].append(train_acc)
    history_stage2['val_loss'].append(val_loss)
    history_stage2['val_acc'].append(val_acc)
    
    epoch_time = time.time() - start_time
    
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print(f"  Time: {epoch_time:.1f}s | LR: {current_lr:.6f}")
    
    # Check if this is the best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
            'class_names': class_names
        }, '../models/best_model.pth')
        print(f"  New best model saved! (Val Acc: {val_acc:.2f}%)")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping triggered after {epoch} epochs")
            print(f"  Best model was at epoch {best_epoch} with {best_val_acc:.2f}%")
            break

print(f"\nStage 2 complete!")
print(f"Best model from epoch {best_epoch} with validation accuracy: {best_val_acc:.2f}%")


STAGE 2: Fine-tuning All Layers
Now I unfreeze everything and train with a lower learning rate.
--------------------------------------------------
  All layers unfrozen for fine-tuning

Epoch 1/20
----------------------------------------
    Batch 20/92 | Loss: 0.9734
    Batch 40/92 | Loss: 1.0730
    Batch 60/92 | Loss: 0.8249
    Batch 80/92 | Loss: 0.7538
  Train Loss: 0.9900 | Train Acc: 69.82%
  Val Loss:   1.1356 | Val Acc:   62.07%
  Time: 188.4s | LR: 0.000100

Epoch 2/20
----------------------------------------
    Batch 20/92 | Loss: 0.7421
    Batch 40/92 | Loss: 0.7484
    Batch 60/92 | Loss: 0.8518
    Batch 80/92 | Loss: 1.1277
  Train Loss: 0.8460 | Train Acc: 75.32%
  Val Loss:   0.9737 | Val Acc:   74.22%
  Time: 203.5s | LR: 0.000100
  New best model saved! (Val Acc: 74.22%)

Epoch 3/20
----------------------------------------
    Batch 20/92 | Loss: 0.6610
    Batch 40/92 | Loss: 1.0485
    Batch 60/92 | Loss: 0.6471
    Batch 80/92 | Loss: 0.7106
  Train Loss: 0.7

In [14]:
# ============================================================================
# STEP 10: SAVE TRAINING HISTORY
# ============================================================================

print("\n" + "=" * 70)
print("STEP 10: SAVING TRAINING HISTORY")
print("=" * 70)

# Combine histories
training_history = {
    'stage1': history_stage1,
    'stage2': history_stage2,
    'best_val_acc': float(best_val_acc),
    'best_epoch': best_epoch,
    'class_names': class_names
}

# Create results folder
os.makedirs('../results', exist_ok=True)

# Save to file
with open('../results/training_history.json', 'w') as f:
    json.dump(training_history, f, indent=2)

print("Saved: results/training_history.json")

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)
print(f"\nBest validation accuracy: {best_val_acc:.2f}%")
print(f"Best model saved at: models/best_model.pth")
print(f"Stage 1 model saved at: models/best_model_stage1.pth")
print(f"Training history saved at: results/training_history.json")

print("\n" + "=" * 70)
print("NEXT: Open Notebook 04 - Model Evaluation")
print("=" * 70)


STEP 10: SAVING TRAINING HISTORY
Saved: results/training_history.json

TRAINING COMPLETE

Best validation accuracy: 80.49%
Best model saved at: models/best_model.pth
Stage 1 model saved at: models/best_model_stage1.pth
Training history saved at: results/training_history.json

NEXT: Open Notebook 04 - Model Evaluation
